Format Dataset for Supervised Fine-Tuning

In this notebook, I will convert the Human Message Rewriter dataset into a supervised fine-tuning format.
Each example currently has:
- instruction
- input
- output

For fine-tuning, we will format each example as a conversation:
- system message: defines the assistant's behavior
- user message: gives the rewriting instruction and input text
- assistant message: provides the target rewritten output

This formatted dataset will later be used to fine-tune the model.

# Importing libraries

In [1]:
import json
from pathlib import Path
import pandas as pd

# Configuration Setup

## Setup Paths

In [2]:
train_path = Path("../data/processed/train.jsonl")
validation_path = Path("../data/processed/validation.jsonl")
test_path = Path("../data/processed/test.jsonl")

print(f"Train file Path : {train_path} | Exists: {train_path.exists()}")
print(f"Validation file Path : {validation_path} | Exists: {validation_path.exists()}")
print(f"Test file Path : {test_path} | Exists: {test_path.exists()}")


Train file Path : ..\data\processed\train.jsonl | Exists: True
Validation file Path : ..\data\processed\validation.jsonl | Exists: True
Test file Path : ..\data\processed\test.jsonl | Exists: True


In [14]:
sft_dir = Path("../data/sft")
sft_dir.mkdir(exist_ok=True)

print(f"SFT directory: {sft_dir} | Exists: {sft_dir.exists()}")

SFT directory: ..\data\sft | Exists: True


In [15]:
train_sft_path = sft_dir / "train_sft.jsonl"
validation_sft_path = sft_dir / "validation_sft.jsonl"
test_sft_path = sft_dir / "test_sft.jsonl"

print(train_sft_path)
print(validation_sft_path)
print(test_sft_path)

..\data\sft\train_sft.jsonl
..\data\sft\validation_sft.jsonl
..\data\sft\test_sft.jsonl


## load train. validation and test splits

In [3]:
def load_jsonl(file_path):

    rows = []

    with file_path.open("r", encoding = "utf-8") as file:
        for line in file:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows

In [5]:
train_rows = load_jsonl(train_path)
validation_rows = load_jsonl(validation_path)
test_rows = load_jsonl(test_path)

print(f"Test rows:",len(train_rows))
print(f"Validation rows:",len(validation_rows))
print(f"Test rows:",len(test_rows))

Test rows: 240
Validation rows: 30
Test rows: 30


# Format dataset

## Test on one cell

In [6]:
example = train_rows[0]
print("Example row from train set:", example)

Example row from train set: {'id': 'ex_0003', 'category': 'workplace', 'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.', 'input': 'Please be informed that I have attached the required document for your reference.', 'output': 'I’ve attached the required document for your reference.', 'source': 'synthetic_curated'}


In [7]:
print("Instruction:")
print(example["instruction"])

print("\nInput:")
print(example["input"])

print("\nOutput:")
print(example["output"])

Instruction:
Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.

Input:
Please be informed that I have attached the required document for your reference.

Output:
I’ve attached the required document for your reference.


## Setup the system message

In [8]:
SYSTEM_MESSAGE = (
    "You are a helpful writing assistant. "
    "Rewrite messages so they sound natural, clear, and human while preserving the original meaning. "
    "Do not add new information."
)

SYSTEM_MESSAGE

'You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.'

## Format as message function

In [9]:
def format_as_message(row):
    user_message = (
    f"Instruction: {row['instruction']}\n\n"
    f"Input: {row['input']}\n\n"
    )

    assistant_message = row["output"]

    return {
        "id": row["id"],
        "category": row["category"],
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

## Single conversation example

In [10]:
formatted_example = format_as_message(example)
print(f"formatted_example: {formatted_example}")

formatted_example: {'id': 'ex_0003', 'category': 'workplace', 'messages': [{'role': 'system', 'content': 'You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.'}, {'role': 'user', 'content': 'Instruction: Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.\n\nInput: Please be informed that I have attached the required document for your reference.\n\n'}, {'role': 'assistant', 'content': 'I’ve attached the required document for your reference.'}]}


In [11]:
for message in formatted_example["messages"]:
    print("=" * 80)
    print("ROLE:", message["role"])
    print(message["content"])

ROLE: system
You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.
ROLE: user
Instruction: Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.

Input: Please be informed that I have attached the required document for your reference.


ROLE: assistant
I’ve attached the required document for your reference.


In [12]:
formatted_preview = [format_as_message(row) for row in train_rows[:5]]

len(formatted_preview)

5

In [13]:
formatted_preview[0]

{'id': 'ex_0003',
 'category': 'workplace',
 'messages': [{'role': 'system',
   'content': 'You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.'},
  {'role': 'user',
   'content': 'Instruction: Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.\n\nInput: Please be informed that I have attached the required document for your reference.\n\n'},
  {'role': 'assistant',
   'content': 'I’ve attached the required document for your reference.'}]}

Conversation Format

We converted raw instruction/input/output rows into a conversation format.

Each formatted example contains:

- a system message that defines the assistant's behavior
- a user message that includes the rewrite instruction and original message
- an assistant message that contains the target rewritten output

This format is useful because instruction-tuned models are already trained to follow chat-style prompts.

## Format all splits

In [16]:
train_sft_rows = [format_as_message(row) for row in train_rows]
validation_sft_rows = [format_as_message(row) for row in validation_rows]
test_sft_rows = [format_as_message(row) for row in test_rows]

print("Train SFT rows:", len(train_sft_rows))
print("Validation SFT rows:", len(validation_sft_rows))
print("Test SFT rows:", len(test_sft_rows))

Train SFT rows: 240
Validation SFT rows: 30
Test SFT rows: 30


# Save Jsonl files and Evaluate

In [17]:
def save_jsonl(rows, file_path):
    with file_path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

In [18]:
save_jsonl(train_sft_rows, train_sft_path)
save_jsonl(validation_sft_rows, validation_sft_path)
save_jsonl(test_sft_rows, test_sft_path)

print("Saved SFT datasets.")

Saved SFT datasets.


In [19]:
print("Train SFT exists:", train_sft_path.exists())
print("Validation SFT exists:", validation_sft_path.exists())
print("Test SFT exists:", test_sft_path.exists())

Train SFT exists: True
Validation SFT exists: True
Test SFT exists: True


Saved SFT Dataset

I formatted and saved the train, validation, and test splits in chat-style supervised fine-tuning format.

Each row now contains:

- id
- category
- messages

The `messages` field contains a system message, user message, and assistant message.

These files will be used later for LoRA/QLoRA supervised fine-tuning.

## load saved sft files

In [20]:


saved_train_sft_rows = load_jsonl(train_sft_path)
saved_validation_sft_rows = load_jsonl(validation_sft_path)
saved_test_sft_rows = load_jsonl(test_sft_path)

print("Saved train SFT rows:", len(saved_train_sft_rows))
print("Saved validation SFT rows:", len(saved_validation_sft_rows))
print("Saved test SFT rows:", len(saved_test_sft_rows))

Saved train SFT rows: 240
Saved validation SFT rows: 30
Saved test SFT rows: 30


##  Validate one SFT row

In [21]:
def validate_sft_row(row):
    errors = []

    if "id" not in row:
        errors.append("Missing id")

    if "category" not in row:
        errors.append("Missing category")

    if "messages" not in row:
        errors.append("Missing messages")
        return errors

    messages = row["messages"]

    if not isinstance(messages, list):
        errors.append("messages must be a list")
        return errors

    if len(messages) != 3:
        errors.append(f"Expected 3 messages, found {len(messages)}")
        return errors

    expected_roles = ["system", "user", "assistant"]

    for index, expected_role in enumerate(expected_roles):
        message = messages[index]

        if "role" not in message:
            errors.append(f"Message {index} missing role")
        elif message["role"] != expected_role:
            errors.append(
                f"Message {index} has role '{message['role']}', expected '{expected_role}'"
            )

        if "content" not in message:
            errors.append(f"Message {index} missing content")
        elif not str(message["content"]).strip():
            errors.append(f"Message {index} has empty content")

    return errors

In [22]:
example_errors = validate_sft_row(saved_train_sft_rows[0])

example_errors

[]

## Validate all SFT splits

In [23]:
def validate_sft_split(rows, split_name):
    split_errors = []

    for index, row in enumerate(rows):
        row_errors = validate_sft_row(row)

        for error in row_errors:
            split_errors.append({
                "split": split_name,
                "row_number": index + 1,
                "id": row.get("id", "missing_id"),
                "error": error
            })

    return split_errors




In [24]:
train_sft_errors = validate_sft_split(saved_train_sft_rows, "train")
validation_sft_errors = validate_sft_split(saved_validation_sft_rows, "validation")
test_sft_errors = validate_sft_split(saved_test_sft_rows, "test")

all_sft_errors = train_sft_errors + validation_sft_errors + test_sft_errors

len(all_sft_errors)

0

In [25]:
if all_sft_errors:
    error_df = pd.DataFrame(all_sft_errors)
    display(error_df.head(20))
else:
    print("No SFT format errors found.")

No SFT format errors found.


## Assistant outputs are not too long or empty

In [27]:
def get_assistant_output(row):
    return row["messages"][2]["content"]

In [28]:



assistant_output_warnings = []

for split_name, rows in [
    ("train", saved_train_sft_rows),
    ("validation", saved_validation_sft_rows),
    ("test", saved_test_sft_rows),
]:
    for index, row in enumerate(rows):
        assistant_output = get_assistant_output(row)
        word_count = len(assistant_output.split())

        if word_count < 3:
            assistant_output_warnings.append({
                "split": split_name,
                "row_number": index + 1,
                "id": row["id"],
                "warning": "Assistant output is very short",
                "assistant_output": assistant_output
            })



## Final SFT Validation report

In [29]:
print("SFT Dataset Validation Report")
print("=" * 40)

print("Rows:")
print(f"- Train: {len(saved_train_sft_rows)}")
print(f"- Validation: {len(saved_validation_sft_rows)}")
print(f"- Test: {len(saved_test_sft_rows)}")

print("\nFormat errors:")
print(f"- Total SFT errors: {len(all_sft_errors)}")

print("\nWarnings:")
print(f"- Very short assistant outputs: {len(assistant_output_warnings)}")

if len(all_sft_errors) == 0:
    print("\nStatus: PASSED")
    print("The SFT dataset is correctly formatted.")
else:
    print("\nStatus: FAILED")
    print("Fix SFT formatting errors before training.")

SFT Dataset Validation Report
Rows:
- Train: 240
- Validation: 30
- Test: 30

Format errors:
- Total SFT errors: 0

Warnings:
- Very short assistant outputs: 0

Status: PASSED
The SFT dataset is correctly formatted.


SFT Format Validation

I have validated the saved supervised fine-tuning datasets.

Each example should contain:

- `id`
- `category`
- `messages`

The `messages` field should contain exactly three messages:

1. system message
2. user message
3. assistant message

This confirms that the dataset is ready for the next stage: supervised fine-tuning.